# Temperature scaling del clasificador K=70 (haversine + augmentation)

Recalibra las probabilidades del modelo entrenado en `cell_classifier_k70_haversine_aug.ipynb` **sin reentrenar**:
los logits se dividen por una temperatura $T$ antes del softmax,

$$p_k = \frac{\exp(z_k / T)}{\sum_m \exp(z_m / T)}.$$

- $T$ es un único parámetro y **no cambia el argmax**: accuracy top-1/top-5 y error en km de la predicción
  directa quedan idénticos.
- Sí cambia todo lo que usa la distribución completa: NLL, calibración (ECE), Brier y, en particular, la
  **predicción por mínimo riesgo** (celda que minimiza $\sum_j p_j\, d(c_j, c_k)$), que puede elegir otra celda.

Para no ajustar y evaluar sobre los mismos datos, la validación global se parte **por panorama** (las 4 vistas de un
panorama quedan del mismo lado) en una mitad de **calibración**, donde se ajusta $T$ minimizando la NLL
(CrossEntropy contra la celda real), y una mitad de **evaluación**, donde se comparan $T=1$ y $T^*$.

Orden: **1. Configuración y datos → 2. Partición → 3. Ajuste de T → 4. Evaluación**.

## 1. Configuración y datos

In [ ]:
from pathlib import Path
from io import BytesIO
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import Dataset as HFDataset, Image as HFImage
from PIL import Image
from scipy.optimize import minimize_scalar
from tqdm.auto import tqdm
from IPython.display import display

# Carpeta de la corrida entrenada (relativa a la raíz del proyecto o absoluta).
RUTA_MODELO = "outputs/resnet18_IMAGENET_celdas_k70_haversine_aug/20261109_180324_467412"
ARCHIVO_CHECKPOINT = "mejor_resnet18_celdas.pt"
RUTA_VALIDACION = "datasets/samples/data/validation/validation.parquet"
COL_CLUSTER = "ind_cluster"
COLS_COORDENADAS = ["latitude", "longitude"]
RADIO_TIERRA_KM = 6371.0
FRACCION_CALIBRACION = 0.5  # Fracción de panoramas de validación usada para ajustar T.
T_MIN, T_MAX = 0.05, 20.0  # Rango de búsqueda de T.
N_BINS_ECE = 10  # Igual que en el notebook de entrenamiento.
BATCH_SIZE = 64
NUM_WORKERS = 0
SEMILLA = 42
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Ejecutá el notebook dentro del proyecto.")


def resolver_ruta(ruta):
    p = Path(ruta).expanduser()
    return (p if p.is_absolute() else raiz / p).resolve()


ruta_modelo = resolver_ruta(RUTA_MODELO)
ruta_checkpoint = ruta_modelo / ARCHIVO_CHECKPOINT
ruta_validacion = resolver_ruta(RUTA_VALIDACION)
ruta_salida = ruta_modelo / "temperature_scaling"
for archivo in (ruta_checkpoint, ruta_modelo / "prototipos.csv", ruta_validacion):
    if not archivo.is_file():
        raise FileNotFoundError(archivo)
assert 0 < FRACCION_CALIBRACION < 1 and 0 < T_MIN < 1 < T_MAX
ruta_salida.mkdir(parents=True, exist_ok=True)
device = torch.device(DISPOSITIVO)
print(f"Modelo: {ruta_checkpoint}\nValidación: {ruta_validacion}\nSalida: {ruta_salida}\nDispositivo: {device}")

### Modelo y validación

Se reconstruye la misma arquitectura (ResNet18 con `Dropout` + `Linear` de 70 salidas) y se cargan los pesos del
mejor checkpoint. Los prototipos salen de la carpeta de la corrida y se controla que su SHA-256 coincida con el
registrado en el checkpoint. La celda real de cada imagen se calcula desde las coordenadas, como en el entrenamiento,
y el preprocesamiento de validación debe ser idéntico al guardado en la configuración.

In [ ]:
checkpoint = torch.load(ruta_checkpoint, map_location="cpu", weights_only=True)
cfg = checkpoint["config"]
if cfg.get("arquitectura") != "resnet18" or cfg.get("tarea") != "clasificacion_celdas_voronoi":
    raise ValueError("El checkpoint no es un clasificador de celdas ResNet18.")

prototipos = pd.read_csv(ruta_modelo / "prototipos.csv").sort_values(COL_CLUSTER).reset_index(drop=True)
NUM_CLASES = len(prototipos)
assert NUM_CLASES == cfg["num_clases"] and (prototipos[COL_CLUSTER].to_numpy() == np.arange(NUM_CLASES)).all()
coords_prototipos = prototipos[["lat", "lon"]].to_numpy(dtype=np.float64)
# Control: si el CSV original de prototipos sigue disponible, debe tener el SHA-256 registrado y las mismas coordenadas.
ruta_proto_original = Path(cfg["prototipos"]["ruta"])
if ruta_proto_original.is_file():
    if hashlib.sha256(ruta_proto_original.read_bytes()).hexdigest() != cfg["prototipos"]["sha256"]:
        raise ValueError(f"{ruta_proto_original} cambió desde el entrenamiento.")
    original = pd.read_csv(ruta_proto_original).sort_values(COL_CLUSTER)[["lat", "lon"]].to_numpy(dtype=np.float64)
    if not np.allclose(original, prototipos[["lat", "lon"]].to_numpy(dtype=np.float64)):
        raise ValueError("Los prototipos de la corrida no coinciden con los del entrenamiento.")


def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.deg2rad, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * RADIO_TIERRA_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


pesos = models.ResNet18_Weights.IMAGENET1K_V1.transforms()
TAMANO_IMAGEN = cfg["tamano_imagen"]
transform_val = transforms.Compose([
    transforms.Resize(TAMANO_IMAGEN),
    transforms.CenterCrop(TAMANO_IMAGEN),
    transforms.ToTensor(),
    transforms.Normalize(pesos.mean, pesos.std),
])
if cfg["preprocesamiento_val"] != repr(transform_val):
    raise ValueError("El preprocesamiento de validación no coincide con el del entrenamiento.")

modelo = models.resnet18(weights=None)
modelo.fc = nn.Sequential(nn.Dropout(cfg["dropout"]), nn.Linear(modelo.fc.in_features, NUM_CLASES))
modelo.load_state_dict(checkpoint["model_state_dict"])
modelo.to(device).eval()
print(f"Checkpoint: época {checkpoint['epoca']} (loss val={checkpoint['loss_validation']:.4f}, "
      f"acc={checkpoint['accuracy_validation']:.3f})")
del checkpoint

datos_val = (HFDataset.from_parquet(str(ruta_validacion), cache_dir=str(raiz / ".cache/modeling/hf_celdas"))
             .cast_column("image", HFImage(decode=False)))
meta = datos_val.select_columns(["image_id", "panoid", *COLS_COORDENADAS]).to_pandas()
lat, lon = (meta[c].to_numpy(dtype=np.float64) for c in COLS_COORDENADAS)
if not (np.isfinite(lat) & np.isfinite(lon)).all():
    raise ValueError("Hay coordenadas inválidas en validación.")
real = (to_xyz(lat, lon) @ to_xyz(coords_prototipos[:, 0], coords_prototipos[:, 1]).T).argmax(axis=1)
# Distancias en km de cada foto a cada prototipo y entre prototipos.
dist_foto_proto = haversine_km(lat[:, None], lon[:, None], coords_prototipos[None, :, 0], coords_prototipos[None, :, 1])
dist_proto = haversine_km(coords_prototipos[:, None, 0], coords_prototipos[:, None, 1],
                          coords_prototipos[None, :, 0], coords_prototipos[None, :, 1])
assert (dist_foto_proto.argmin(axis=1) == real).all()
print(f"Validación: {len(meta):,} imágenes, {meta['panoid'].nunique():,} panoramas, "
      f"{len(np.unique(real))} celdas con imágenes")

### Logits de validación

La inferencia se hace una sola vez (dropout desactivado, `eval`) y los logits se guardan en
`temperature_scaling/logits_validation.npz`. Si el archivo ya existe y corresponde a las mismas imágenes y al
mismo checkpoint, se reutiliza.

In [ ]:
class ImagenesValidacion(Dataset):
    def __init__(self, datos, transform):
        self.datos, self.transform = datos, transform

    def __len__(self):
        return len(self.datos)

    def __getitem__(self, idx):
        contenido = self.datos[int(idx)]["image"]
        fuente = BytesIO(contenido["bytes"]) if contenido.get("bytes") is not None else resolver_ruta(contenido["path"])
        with Image.open(fuente) as imagen:
            return self.transform(imagen.convert("RGB"))


ruta_logits = ruta_salida / "logits_validation.npz"
ids = meta["image_id"].astype(str).to_numpy()
mtime_ckpt = ruta_checkpoint.stat().st_mtime_ns
logits = None
if ruta_logits.is_file():
    guardado = np.load(ruta_logits, allow_pickle=False)
    if np.array_equal(guardado["image_id"], ids) and int(guardado["mtime_checkpoint"]) == mtime_ckpt:
        logits = guardado["logits"]
        print(f"Logits reutilizados de {ruta_logits}")
if logits is None:
    loader = DataLoader(ImagenesValidacion(datos_val, transform_val), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
    salidas = []
    with torch.inference_mode():
        for imagenes in tqdm(loader, desc="Inferencia"):
            salidas.append(modelo(imagenes.to(device, non_blocking=True)).float().cpu())
    logits = torch.cat(salidas).numpy().astype(np.float64)
    np.savez(ruta_logits, logits=logits, image_id=ids, mtime_checkpoint=mtime_ckpt)
    print(f"Logits guardados en {ruta_logits}")
assert logits.shape == (len(meta), NUM_CLASES)
print(f"Accuracy top-1 (control, debe coincidir con el entrenamiento): {(logits.argmax(1) == real).mean():.4f}")

## 2. Partición calibración / evaluación

Se sortean panoramas (no imágenes): todas las vistas de un panorama quedan en la misma mitad, así la evaluación no
contiene fotos casi idénticas a las usadas para ajustar $T$.

In [ ]:
rng = np.random.default_rng(SEMILLA)
panoramas = meta["panoid"].unique()
pano_calib = set(rng.choice(panoramas, size=int(round(FRACCION_CALIBRACION * len(panoramas))), replace=False))
es_calib = meta["panoid"].isin(pano_calib).to_numpy()
idx_calib, idx_eval = np.flatnonzero(es_calib), np.flatnonzero(~es_calib)
assert not set(meta["panoid"].iloc[idx_calib]) & set(meta["panoid"].iloc[idx_eval])
print(f"Calibración: {len(idx_calib):,} imágenes ({len(pano_calib):,} panoramas)")
print(f"Evaluación:  {len(idx_eval):,} imágenes ({len(panoramas) - len(pano_calib):,} panoramas)")
print(f"Accuracy top-1 por mitad (T no la cambia): calibración {(logits[idx_calib].argmax(1) == real[idx_calib]).mean():.3f}, "
      f"evaluación {(logits[idx_eval].argmax(1) == real[idx_eval]).mean():.3f}")

## 3. Ajuste de T

$T^* = \arg\min_T \; \text{NLL}_{\text{calibración}}(T)$, con la NLL = CrossEntropy contra la celda real (la misma
"CE dura" que reporta el notebook de entrenamiento). Se optimiza sobre $\log T$ (una sola variable, función suave).
El gráfico muestra la NLL en función de $T$ en ambas mitades: si los mínimos coinciden, $T^*$ generaliza.

In [ ]:
def softmax_T(z, T):
    z = z / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def nll(z, y, T):
    z = z / T
    z = z - z.max(axis=1, keepdims=True)
    log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    return float(-log_p[np.arange(len(y)), y].mean())


z_cal, y_cal = logits[idx_calib], real[idx_calib]
z_ev, y_ev = logits[idx_eval], real[idx_eval]
res = minimize_scalar(lambda log_t: nll(z_cal, y_cal, np.exp(log_t)),
                      bounds=(np.log(T_MIN), np.log(T_MAX)), method="bounded", options={"xatol": 1e-6})
T_OPT = float(np.exp(res.x))
if not (T_MIN * 1.01 < T_OPT < T_MAX * 0.99):
    print("AVISO: T* quedó en el borde del rango de búsqueda; ampliá T_MIN/T_MAX.")
print(f"T* = {T_OPT:.4f}  (NLL calibración: T=1 → {nll(z_cal, y_cal, 1.0):.4f}, T* → {nll(z_cal, y_cal, T_OPT):.4f})")

grilla_T = np.exp(np.linspace(np.log(0.3), np.log(4.0), 80))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(grilla_T, [nll(z_cal, y_cal, t) for t in grilla_T], label="calibración")
ax.plot(grilla_T, [nll(z_ev, y_ev, t) for t in grilla_T], label="evaluación")
ax.axvline(1.0, color="gray", linestyle=":", label="T = 1")
ax.axvline(T_OPT, color="black", linestyle="--", label=f"T* = {T_OPT:.2f}")
ax.set(xscale="log", xlabel="Temperatura T", ylabel="NLL (CE contra la celda real)", title="NLL en función de T")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ruta_salida / "nll_vs_T.png", dpi=160)
plt.show()

## 4. Evaluación sobre la mitad de evaluación

Comparación $T=1$ vs. $T^*$ sobre imágenes que no participaron del ajuste:

- **NLL**, **ECE** (top-label, `N_BINS_ECE` bins, igual que en el entrenamiento), **Brier** multiclase y confianza
  media frente a la accuracy.
- **Accuracy top-1/top-5** y error en km del **argmax**: deben quedar idénticos (control).
- **Predicción por mínimo riesgo** (`_mbr`): accuracy y error medio/mediano en km, que sí pueden cambiar.

In [ ]:
def ece_top_label(p, y, n_bins=N_BINS_ECE):
    conf, acierto = p.max(axis=1), p.argmax(axis=1) == y
    bin_id = np.clip(np.digitize(conf, np.linspace(0, 1, n_bins + 1)[1:-1]), 0, n_bins - 1)
    return float(sum((bin_id == b).mean() * abs(acierto[bin_id == b].mean() - conf[bin_id == b].mean())
                     for b in range(n_bins) if (bin_id == b).any()))


def metricas(z, idx, T):
    y = real[idx]
    p = softmax_T(z, T)
    pred = p.argmax(axis=1)
    pred_mbr = (p @ dist_proto).argmin(axis=1)
    err = dist_foto_proto[idx, pred]
    err_mbr = dist_foto_proto[idx, pred_mbr]
    top5 = np.argsort(-p, axis=1)[:, :5]
    onehot = np.eye(NUM_CLASES)[y]
    return {
        "T": T,
        "nll": nll(z, y, T),
        "ece": ece_top_label(p, y),
        "brier": float(((p - onehot) ** 2).sum(axis=1).mean()),
        "confianza_media": float(p.max(axis=1).mean()),
        "accuracy_top1": float((pred == y).mean()),
        "accuracy_top5": float((top5 == y[:, None]).any(axis=1).mean()),
        "error_medio_km": float(err.mean()),
        "error_mediano_km": float(np.median(err)),
        "accuracy_top1_mbr": float((pred_mbr == y).mean()),
        "error_medio_km_mbr": float(err_mbr.mean()),
        "error_mediano_km_mbr": float(np.median(err_mbr)),
    }


tabla = pd.DataFrame({"T = 1": metricas(z_ev, idx_eval, 1.0), f"T* = {T_OPT:.2f}": metricas(z_ev, idx_eval, T_OPT)})
tabla["diferencia"] = tabla.iloc[:, 1] - tabla.iloc[:, 0]
display(tabla.round(4))
assert tabla.loc["accuracy_top1", "diferencia"] == 0 and tabla.loc["error_medio_km", "diferencia"] == 0

resultado = {
    "checkpoint": str(ruta_checkpoint), "T_opt": T_OPT, "criterio": "minimizar NLL (CE dura) en calibración",
    "fraccion_calibracion": FRACCION_CALIBRACION, "particion": "por panoid", "semilla": SEMILLA,
    "n_calibracion": int(len(idx_calib)), "n_evaluacion": int(len(idx_eval)),
    "evaluacion_T1": metricas(z_ev, idx_eval, 1.0), "evaluacion_Topt": metricas(z_ev, idx_eval, T_OPT),
}
(ruta_salida / "temperature_scaling.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
tabla.to_csv(ruta_salida / "metricas_evaluacion.csv")
print(f"Guardado en {ruta_salida}")

### Diagrama de confiabilidad

Accuracy por bin de confianza (top-label) en la mitad de evaluación, antes y después de recalibrar. Por debajo de
la diagonal: sobreconfiado; por encima: subconfiado. Las barras grises muestran cuántas imágenes caen en cada bin.

In [ ]:
def curva_confiabilidad(p, y, n_bins=N_BINS_ECE):
    conf, acierto = p.max(axis=1), p.argmax(axis=1) == y
    bin_id = np.clip(np.digitize(conf, np.linspace(0, 1, n_bins + 1)[1:-1]), 0, n_bins - 1)
    filas = [(conf[bin_id == b].mean(), acierto[bin_id == b].mean(), (bin_id == b).sum())
             for b in range(n_bins) if (bin_id == b).any()]
    return np.array(filas).T


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, T, titulo in ((axes[0], 1.0, "T = 1 (sin recalibrar)"), (axes[1], T_OPT, f"T* = {T_OPT:.2f}")):
    p = softmax_T(z_ev, T)
    c, a, n = curva_confiabilidad(p, y_ev)
    ax.plot([0, 1], [0, 1], "--", color="gray", label="calibración perfecta")
    ax.plot(c, a, marker="o", label=f"modelo (ECE = {ece_top_label(p, y_ev):.3f})")
    ax2 = ax.twinx()
    ax2.bar(c, n, width=0.06, color="lightgray", alpha=0.6, zorder=0)
    ax2.set_ylabel("imágenes por bin")
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)
    ax.set(xlabel="confianza declarada (prob. máxima)", ylabel="accuracy", title=titulo, xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ruta_salida / "confiabilidad_T1_vs_Topt.png", dpi=160)
plt.show()

### Efecto de T sobre la predicción por mínimo riesgo

El argmax no depende de $T$, pero la predicción por mínimo riesgo sí: con $T$ alto la distribución se aplana y la
regla tiende a elegir celdas "intermedias"; con $T$ bajo se acerca al argmax. Se muestra en la mitad de evaluación
como diagnóstico (no se usa para elegir $T$, que se fijó solo con la NLL de calibración).

In [ ]:
grilla_mbr = np.exp(np.linspace(np.log(0.3), np.log(4.0), 40))
curva = pd.DataFrame([metricas(z_ev, idx_eval, t) for t in grilla_mbr])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, titulo in ((axes[0], "error_medio_km_mbr", "Error medio (km) — mínimo riesgo"),
                        (axes[1], "error_mediano_km_mbr", "Error mediano (km) — mínimo riesgo"),
                        (axes[2], "accuracy_top1_mbr", "Accuracy top-1 — mínimo riesgo")):
    ax.plot(curva["T"], curva[col], label="mínimo riesgo")
    base = {"error_medio_km_mbr": "error_medio_km", "error_mediano_km_mbr": "error_mediano_km",
            "accuracy_top1_mbr": "accuracy_top1"}[col]
    ax.axhline(curva[base].iloc[0], color="tab:red", linestyle="-.", label="argmax (no depende de T)")
    ax.axvline(1.0, color="gray", linestyle=":", label="T = 1")
    ax.axvline(T_OPT, color="black", linestyle="--", label=f"T* = {T_OPT:.2f}")
    ax.set(xscale="log", xlabel="Temperatura T", title=titulo)
    ax.grid(alpha=0.3)
axes[0].legend()
fig.tight_layout()
fig.savefig(ruta_salida / "minimo_riesgo_vs_T.png", dpi=160)
plt.show()
curva.to_csv(ruta_salida / "minimo_riesgo_vs_T.csv", index=False)